Exercise 1- Conceptual Questions

1. Learning Rate (alpha)

If alpha = 0, the Q-values will never update, meaning the agent will not learn anything
If alpha = 1, the agent will completely replace old knowledge with new information after each step, which may cause instability
The learning rate controls how much new information overrides old knowledge


2. Q-Table Initialization

If the Q-table is initialized with large positive values, the agent will be encouraged to explore more
This is because all actions initially look very good, so the agent will try them out to verify their actual rewards


3. TD vs Monte Carlo

The TD approach updates the model after every step, while Monte Carlo updates only at the end of the episode
TD is better for long games like chess because it learns faster and does not need to wait until the end of the game

In [2]:
import gym
import numpy as np

env = gym.make('FrozenLake-v1', is_slippery=False)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
state_size = env.observation_space.n
action_size = env.action_space.n

Q = np.zeros((state_size, action_size))

print("Q-table shape:", Q.shape)

Q-table shape: (16, 4)


In [4]:
alpha = 0.1
gamma = 0.99
epsilon = 0.1
episodes = 10000

In [6]:
for episode in range(episodes):
    state, _ = env.reset()
    done = False

    while not done:
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state])

        new_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        Q[state, action] = Q[state, action] + alpha * (
            reward + gamma * np.max(Q[new_state]) - Q[state, action]
        )

        state = new_state

In [7]:
print("Final Q-table:")
print(Q)

Final Q-table:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


The Q-table shows the learned values for each state-action pair
States closer to the goal have higher values, especially for actions that move toward the goal
This means the agent has learned the optimal path

In [8]:
min_epsilon = 0.01
max_epsilon = 1.0
decay_rate = 0.001

Q_decay = np.zeros((state_size, action_size))
rewards = []

for episode in range(episodes):
    state, _ = env.reset()
    done = False
    total_reward = 0

    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)

    while not done:
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q_decay[state])

        new_state, reward, done, _, _ = env.step(action)

        Q_decay[state, action] = Q_decay[state, action] + alpha * (
            reward + gamma * np.max(Q_decay[new_state]) - Q_decay[state, action]
        )

        state = new_state
        total_reward += reward

    rewards.append(total_reward)

At the beginning, the agent explores more due to high epsilon
Over time, epsilon decreases, and the agent exploits learned knowledge
This leads to faster learning and better final performance compared to constant epsilon